# Classification 
This notebook trains a baseline text classifier with a leakage-safe split strategy.

## Leakage controls
- Uses only training-safe data from `data/labeled/dataset.csv`
- Uses grouped + stratified splitting via `doc_id` to prevent document overlap across splits
- Verifies zero `text_hash` overlap across train/validation/test

In [1]:
import os
import re
import hashlib
from pathlib import Path
import joblib
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

cwd = Path.cwd()
candidate_roots = [cwd, cwd.parent]
candidate_roots.extend([p for p in cwd.iterdir() if p.is_dir()])
project_root = None
for c in candidate_roots:
    if (c / 'setup.py').exists() and (c / 'notebooks').exists():
        project_root = c
        break
if project_root is None:
    raise RuntimeError('Could not locate project root containing setup.py and notebooks/')
os.chdir(project_root)

def make_template_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', ' <num> ', text)
    text = re.sub(r'\b[a-z]\b', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df = pd.read_csv('data/labeled/dataset.csv')

required_cols = {'text', 'label'}
missing = required_cols.difference(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df = df.dropna(subset=['text', 'label']).copy()
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != ''].reset_index(drop=True)

if 'text_hash' not in df.columns:
    df['text_hash'] = df['text'].map(lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest())

if 'doc_id' not in df.columns:
    if {'source', 'source_id'}.issubset(df.columns):
        df['doc_id'] = df['source'].astype(str) + '::' + df['source_id'].astype(str)
    else:
        df['doc_id'] = df['text_hash']

if 'template_hash' not in df.columns:
    df['template_hash'] = df['text'].map(make_template_text).map(lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest())

df['split_group'] = df['doc_id'].astype(str) + '::' + df['template_hash'].astype(str)

print(f'Dataset rows: {len(df)}')
print(df['label'].value_counts())

Dataset rows: 2734
label
news        850
email       850
invoice     609
contract    425
Name: count, dtype: int64


In [2]:
# Grouped + stratified split: train/val/test = 64/16/20 (approx)
sgkf_outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
outer_split = next(sgkf_outer.split(df['text'], df['label'], groups=df['split_group']))
train_val_idx, test_idx = outer_split

df_train_val = df.iloc[train_val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

sgkf_inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
inner_split = next(
    sgkf_inner.split(df_train_val['text'], df_train_val['label'], groups=df_train_val['split_group'])
)
train_idx, val_idx = inner_split

df_train = df_train_val.iloc[train_idx].reset_index(drop=True)
df_val = df_train_val.iloc[val_idx].reset_index(drop=True)

print('Split sizes:')
print(f'Train: {len(df_train)}')
print(f'Val:   {len(df_val)}')
print(f'Test:  {len(df_test)}')

Split sizes:
Train: 1641
Val:   547
Test:  546


In [3]:
def overlap_count(a, b):
    return len(set(a).intersection(set(b)))

print('Split overlap checks:')
print(f"text_hash train ∩ val:      {overlap_count(df_train['text_hash'], df_val['text_hash'])}")
print(f"text_hash train ∩ test:     {overlap_count(df_train['text_hash'], df_test['text_hash'])}")
print(f"text_hash val ∩ test:       {overlap_count(df_val['text_hash'], df_test['text_hash'])}")
print(f"template_hash train ∩ val:  {overlap_count(df_train['template_hash'], df_val['template_hash'])}")
print(f"template_hash train ∩ test: {overlap_count(df_train['template_hash'], df_test['template_hash'])}")
print(f"template_hash val ∩ test:   {overlap_count(df_val['template_hash'], df_test['template_hash'])}")

assert overlap_count(df_train['text_hash'], df_val['text_hash']) == 0
assert overlap_count(df_train['text_hash'], df_test['text_hash']) == 0
assert overlap_count(df_val['text_hash'], df_test['text_hash']) == 0
assert overlap_count(df_train['template_hash'], df_val['template_hash']) == 0
assert overlap_count(df_train['template_hash'], df_test['template_hash']) == 0
assert overlap_count(df_val['template_hash'], df_test['template_hash']) == 0

Split overlap checks:
text_hash train ∩ val:      0
text_hash train ∩ test:     0
text_hash val ∩ test:       0
template_hash train ∩ val:  0
template_hash train ∩ test: 0
template_hash val ∩ test:   0


In [4]:
X_train = df_train['text']
y_train = df_train['label']

X_val = df_val['text']
y_val = df_val['label']

X_test = df_test['text']
y_test = df_test['label']

## Logistic regression

In [5]:
lr_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=80000
    )),
    ('model', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ))
])

lr_model.fit(X_train, y_train)

lr_val_pred = lr_model.predict(X_val)

print('Logistic Regression - Validation')
print('Accuracy:', accuracy_score(y_val, lr_val_pred))
print('Macro F1:', f1_score(y_val, lr_val_pred, average='macro'))
print(classification_report(y_val, lr_val_pred, digits=4))

Logistic Regression - Validation
Accuracy: 0.9835466179159049
Macro F1: 0.983733348276445
              precision    recall  f1-score   support

    contract     1.0000    0.9529    0.9759        85
       email     0.9708    0.9765    0.9736       170
     invoice     1.0000    1.0000    1.0000       122
        news     0.9769    0.9941    0.9854       170

    accuracy                         0.9835       547
   macro avg     0.9869    0.9809    0.9837       547
weighted avg     0.9837    0.9835    0.9835       547



## Linear SVM

In [6]:
svm_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=80000
    )),
    ('model', LinearSVC(
        class_weight='balanced',
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

svm_val_pred = svm_model.predict(X_val)

print('Linear SVM - Validation')
print('Accuracy:', accuracy_score(y_val, svm_val_pred))
print('Macro F1:', f1_score(y_val, svm_val_pred, average='macro'))
print(classification_report(y_val, svm_val_pred, digits=4))

Linear SVM - Validation
Accuracy: 0.9945155393053017
Macro F1: 0.9948507083341414
              precision    recall  f1-score   support

    contract     1.0000    0.9882    0.9941        85
       email     0.9883    0.9941    0.9912       170
     invoice     1.0000    1.0000    1.0000       122
        news     0.9941    0.9941    0.9941       170

    accuracy                         0.9945       547
   macro avg     0.9956    0.9941    0.9949       547
weighted avg     0.9945    0.9945    0.9945       547



## Multinomial Naive Bayes

In [7]:
nb_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=80000
    )),
    ('model', MultinomialNB())
])

nb_model.fit(X_train, y_train)

nb_val_pred = nb_model.predict(X_val)

print('Multinomial Naive Bayes - Validation')
print('Accuracy:', accuracy_score(y_val, nb_val_pred))
print('Macro F1:', f1_score(y_val, nb_val_pred, average='macro'))
print(classification_report(y_val, nb_val_pred, digits=4))

Multinomial Naive Bayes - Validation
Accuracy: 0.9725776965265083
Macro F1: 0.9771658145011972
              precision    recall  f1-score   support

    contract     1.0000    0.9882    0.9941        85
       email     0.9936    0.9176    0.9541       170
     invoice     1.0000    1.0000    1.0000       122
        news     0.9239    1.0000    0.9605       170

    accuracy                         0.9726       547
   macro avg     0.9794    0.9765    0.9772       547
weighted avg     0.9744    0.9726    0.9725       547

